# Jev-CeNN — Global + Local Attention Repeated Twice: Faster 600-Step Selective 3→1 Lab

[![Run in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Jev_CeNN_GlobalLocal_x2_600Step_Colab.ipynb)

## Objective and motivation

The earlier selective W512 two-stage experiment reached only **0.96557 cosine** for group 0–2 at 450 distillation steps. Full-model recovery decreased agreement from **92.9167% at step 1** to **92.0833% at steps 20/40**, so adding more of the same costly end-to-end training is unlikely to help without altering the architecture and selection.

**New architecture, per original three-layer group:**

\`\`\`text
Teacher:       [Global Attention + FFN] -> [Sliding Attention + FFN] -> [Sliding Attention + FFN]
New candidate: [FULL attention] -> [sliding W129] -> [CeNN + FFN]
            -> [FULL attention] -> [sliding W129] -> [CeNN + FFN]
\`\`\`

Exactly **four sequential attention passes** (global/local/global/local), two multiscale cellular residuals, two feedforward networks, and a zero-initialized optional rank-64 correction. This entire candidate occupies ONE ModernBERT encoder slot; the two subsequent native teacher blocks are full-layer bypasses *only when accepted*. **Global really means all valid tokens** (not a centered W512 mask that fails to reach the far edge). The new block does MORE attention work than the three original layers; any GPU speedup is a hypothesis, not an assumption.

## 600 optimizer updates maximum per candidate

- Stage A: **520 local feature-distillation optimizer updates**, with precomputed/cached teacher triple inputs and targets and actual *current student-prefix states*. Unlike the previous Colab, each local update does **NOT** run the frozen 28-layer teacher and student encoder.
- Stage B: **80 end-to-end optimizer updates** through the evolving student, training ONLY the candidate; prior accepted groups, remaining native layers and task head are frozen. Early stopping selects a DEV-GATE checkpoint and prevents damage from recovery.
- **TOTAL 600 updates per attempted group**, including recovery; no hidden CeNN pretraining updates. CeNN gains and weights learn during both stages.
- Use native SDPA for all four attention stages, no sparse Python K/V gather, FP32 by default for stability. Training features are cached only for the *current* group and released before training the next candidate.

## Scientifically selective

Each candidate is evaluated against the original teacher and the **already compressed current student**. Retain candidate only if **real student-prefix local cosine ≥0.99**, NMSE ≤0.025, cumulative teacher decision agreement ≥0.99 and gold drop ≤1.5 percentage points. Otherwise restore the original three full encoder blocks. Optionally require measured GPU speedup.

Data: case-disjoint TRAIN (700), DEV-PROBE (80), DEV-GATE (140), final sealed HOLDOUT (200), all sourced from official dataset TRAIN. Original benchmark TEST is not accessed. The final HOLDOUT is evaluated **once after all accept/reject decisions**, not used for tuning. Since teacher pretrained on source benchmark TRAIN, this holdout is independent only of the present experiment, not necessarily teacher pretraining.

The notebook saves a complete JSON/CSV report, candidate checkpoints, selective acceptance reasons, paired B1/B2 short/long CUDA timings and an audit figure. **No automatic model publishing or claims of 0.99 convergence before GPU execution.**


In [ ]:
#@title 1. Install pinned teacher and latest GlobalLocal×2 CeNN dependencies
%pip -q install "transformers==4.57.6" "datasets>=2.20,<5" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"


In [ ]:
#@title 2. Four attention passes, strict 600-update budget and original teacher
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from transformers.models.modernbert.modeling_modernbert import apply_rotary_pos_emb
from tinycenn_lm.laya_lab.pdelta import PDelta3GDN2CLVRAttention
from tinycenn_lm.laya_lab.core import _depthwise_sequence_conv, _valid_tokens
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence, collate_items,
    normalize_question,
)
from tinycenn_lm.decision_training import require_finite, decision_distill_loss, guarded_step
from tinycenn_lm.sliding_window_fusion import SlidingWindowFusion

SEED=2028
SOURCE_MODEL="convaiinnovations/laya-typed-decisions" #@param {type:"string"}
MAX_LEN=512
FULL_ATTENTION_CONTEXT=512 #@param {type:"integer"}
LOCAL_WINDOW=129 #@param {type:"integer"}
CORRECTION_RANK=64 #@param {type:"integer"}
TRAIN_CASES=700 #@param {type:"integer"}
DEV_CASES=220 #@param {type:"integer"}
HOLDOUT_CASES=200 #@param {type:"integer"}
PROBE_CASES=80 #@param {type:"integer"}
BATCH=2 #@param {type:"integer"}
PROBE_BATCHES=12 #@param {type:"integer"}
DEV_GATE_DECISIONS=240 #@param {type:"integer"}
HOLDOUT_DECISIONS=500 #@param {type:"integer"}
TOTAL_OPTIMIZER_STEPS=600 #@param {type:"integer"}
LOCAL_DISTILL_STEPS=520 #@param {type:"integer"}
FULL_MODEL_RECOVERY_STEPS=80 #@param {type:"integer"}
CACHE_BATCHES=96 #@param {type:"integer"}
PROBE_EVERY=65 #@param {type:"integer"}
FEEDBACK_EVERY=20 #@param {type:"integer"}
COSINE_WEIGHT=3.0
PAIR_INTERMEDIATE_WEIGHT=0.20
INTERMEDIATE_WEIGHT=0.10
CENN_AUX_WEIGHT=0.02
GROUP_LR=0.00015
QKV_LR=0.000015
FEEDBACK_LR=0.000004 #@param {type:"number"}
HIDDEN_ALIGN_WEIGHT=0.05 #@param {type:"number"}
TARGET_COSINE=0.99
TARGET_NMSE=0.025
TARGET_AGREEMENT=0.99
MAX_GOLD_DROP=0.015
MAX_INCREMENTAL_AGREEMENT_DROP=0.01
REQUIRE_CANDIDATE_SPEEDUP=False #@param {type:"boolean"}
USE_FP16_AMP=False #@param {type:"boolean"}
BENCH_REPEATS=10 #@param {type:"integer"}
OUTPUT_DIR=Path("/content/Jev_CeNN_GlobalLocal_x2_600Step")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
assert MAX_LEN==512 and FULL_ATTENTION_CONTEXT==512 and BATCH>0
assert LOCAL_WINDOW==129
assert TOTAL_OPTIMIZER_STEPS==LOCAL_DISTILL_STEPS+FULL_MODEL_RECOVERY_STEPS==600
assert CACHE_BATCHES>=PROBE_BATCHES
assert TRAIN_CASES>0 and DEV_CASES>PROBE_CASES>0 and HOLDOUT_CASES>0
assert 0<HOLDOUT_DECISIONS and 0<DEV_GATE_DECISIONS
assert PROBE_BATCHES>=2 and LOCAL_DISTILL_STEPS>=1
assert 0<=FULL_MODEL_RECOVERY_STEPS
assert PROBE_EVERY>=1 and FEEDBACK_EVERY>=1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs a CUDA GPU (Colab > Runtime > Change runtime type).")
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda")
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

source_dir = Path(snapshot_download(
    SOURCE_MODEL,
    allow_patterns=["rl_agent_config.json", "model.safetensors", "encoder/*", "tokenizer/*"],
))
teacher, source_cfg = build_checkpoint_model(source_dir)
student_eval, _ = build_checkpoint_model(source_dir)
tokenizer = AutoTokenizer.from_pretrained(str(source_dir / "tokenizer"))
teacher.to(device).eval().requires_grad_(False)
student_eval.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile = False
student_eval.encoder.config.reference_compile = False

print("GPU:", torch.cuda.get_device_name(0), "| AMP:", amp_dtype)
from tinycenn_lm.standalone_decision import full_attention_indices
GLOBAL_LAYERS = full_attention_indices(teacher)
SLIDING_LAYERS = [i for i in range(len(teacher.encoder.layers)) if i not in GLOBAL_LAYERS]
assert len(teacher.encoder.layers)==28 and GLOBAL_LAYERS==list(range(0,28,3))
for i in GLOBAL_LAYERS:
    assert tuple(teacher.encoder.layers[i].attn.local_attention) == (-1, -1)
print("Global layers:", GLOBAL_LAYERS, "| native sliding:", SLIDING_LAYERS)
print("Four attention passes: FULL -> W",LOCAL_WINDOW," -> FULL -> W",LOCAL_WINDOW,"| total optimizer updates:",TOTAL_OPTIMIZER_STEPS)
assert all(tuple(teacher.encoder.layers[i].attn.local_attention)==(64,64) for i in SLIDING_LAYERS)
print("Selective groups; all rejected groups stay native. Final HOLDOUT is sealed.")
print("Teacher:", SOURCE_MODEL, "| official test split stays untouched")


In [ ]:
#@title 3. Partition dataset by CASE: TRAIN, PROBE, DEV-GATE and sealed HOLDOUT
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

from tinycenn_lm.jev_cenn_selective import (
    verify_case_partition, judge_replacement, plan_active_layers,
)
rows=list(load_dataset("LocalLLaMA/typed-decisions","all",split="train"))
rng=random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES+DEV_CASES+HOLDOUT_CASES<=len(rows), (
    f"Required TRAIN+DEV+HOLDOUT cases exceeds {len(rows)} dataset TRAIN cases")
train_rows=rows[:TRAIN_CASES]
dev_rows=rows[TRAIN_CASES:TRAIN_CASES+DEV_CASES]
holdout_rows=rows[TRAIN_CASES+DEV_CASES:TRAIN_CASES+DEV_CASES+HOLDOUT_CASES]
split_info=verify_case_partition(train_rows,dev_rows,holdout_rows)
probe_rows=dev_rows[:PROBE_CASES]
gate_rows=dev_rows[PROBE_CASES:]
verify_case_partition(train_rows,probe_rows,gate_rows)
assert not {str(x["id"]) for x in holdout_rows}&{
    str(x["id"]) for x in dev_rows}
train_items=make_items_from_rows(train_rows,SEED,True)
probe_items=make_items_from_rows(probe_rows,SEED+1,True)
gate_items=make_items_from_rows(gate_rows,SEED+2,True)
# IMPORTANT: holdout_items are NOT constructed yet. The final evaluation
# cell materializes these previously sealed rows after all decisions are made.
assert len(gate_items)>=DEV_GATE_DECISIONS
assert len(probe_items)>=PROBE_BATCHES*BATCH
print("Independent partitions:",split_info,
      "probe cases",len(probe_rows),"gate cases",len(gate_rows))
print("Optimizer/probe/gate decisions:",len(train_items),
      len(probe_items),len(gate_items))

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

# Only case-isolated TRAIN / PROBE / DEV-GATE inputs are available here.
# The independent HOLDOUT remains sealed until the final evaluation cell.
print("TRAIN / PROBE / DEV-GATE decisions:",
      len(train_items), len(probe_items), len(gate_items))
print("Official TEST untouched; independent HOLDOUT not materialized.")


In [ ]:
#@title 4. Enumerate the nine prospective 3→1 groups (none forced)
from tinycenn_lm.jev_cenn_full_encoder import (
    group_plan, preserved_tail, install_group, restore_group
)
from tinycenn_lm.jev_cenn_full_local_repeat import (
    FullLocalRepeatCeNNSuperLayer, sdpa_full_or_window
)
from tinycenn_lm.jev_cenn_selective import (
    judge_replacement, plan_active_layers
)
GROUPS=group_plan(len(teacher.encoder.layers))
assert GROUPS==tuple((i,i+1,i+2) for i in range(0,27,3))
assert preserved_tail(28)==(27,)
assert all(g[0] in GLOBAL_LAYERS for g in GROUPS)
assert all(g[1] in SLIDING_LAYERS and g[2] in SLIDING_LAYERS for g in GROUPS)
assert LOCAL_WINDOW==129
print("Candidate triples:",GROUPS)
print("Exact attention topology: FULL, W129, FULL, W129")
print("Budget per triple:",LOCAL_DISTILL_STEPS,"local +",
      FULL_MODEL_RECOVERY_STEPS,"changed-model =",TOTAL_OPTIMIZER_STEPS)
